# Periodontal Survey Analysis Report

This notebook presents a research-style exploratory analysis of the periodontal survey among IT professionals. The augmented `plus_generated` dataset is used for primary visuals and exploratory inference, while the observed-only dataset is loaded again for sensitivity checks.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, SVG, display

def find_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'periodontal_survey_mar_8_cutoff.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.run_analysis import run_pipeline

AUGMENTED_CSV = ROOT / 'periodontal_survey_mar_8_cutoff_plus_generated.csv'
OBSERVED_CSV = ROOT / 'periodontal_survey_mar_8_cutoff.csv'
OUT_DIR = ROOT / 'outputs'
TABLES_DIR = OUT_DIR / 'tables'
FIGURES_DIR = OUT_DIR / 'figures'
FORCE_REGENERATE = False

required_outputs = [
    OUT_DIR / 'cleaned_analysis_dataset.csv',
    TABLES_DIR / 'analysis_metadata.json',
    TABLES_DIR / 'insights.md',
]

if FORCE_REGENERATE or not all(path.exists() for path in required_outputs):
    run_pipeline(AUGMENTED_CSV, OBSERVED_CSV, OUT_DIR)


## Data provenance

The notebook reconstructs the score variables directly from item responses. This is necessary because the source app writes a `Knowledge Score` field but does not actually compute it.

In [ ]:
metadata = json.loads((TABLES_DIR / 'analysis_metadata.json').read_text())
metadata

## Analysis-ready dataset

The cleaned dataset adds `sample_source`, recomputed score columns, response labels, locality normalization, and a duration artifact flag.

In [ ]:
analysis_df = pd.read_csv(OUT_DIR / 'cleaned_analysis_dataset.csv', parse_dates=['timestamp'])
analysis_df[['sample_source', 'age_range', 'gender', 'professional_experience', 'work_mode', 'previous_treatment', 'knowledge_score', 'attitude_score', 'practice_index']].head()

## Descriptive tables

In [ ]:
table_one = pd.read_csv(TABLES_DIR / 'table_1_sample_characteristics.csv')
composite_summary = pd.read_csv(TABLES_DIR / 'composite_score_summary.csv')
reliability_summary = pd.read_csv(TABLES_DIR / 'reliability_summary.csv')
display(table_one.head(20))
display(composite_summary)
display(reliability_summary)

## Statistical analysis outputs

In [ ]:
subgroup_results = pd.read_csv(TABLES_DIR / 'subgroup_results_augmented.csv')
pairwise_results = pd.read_csv(TABLES_DIR / 'subgroup_pairwise_results_augmented.csv')
kap_correlations = pd.read_csv(TABLES_DIR / 'kap_correlations.csv')
regression_results = pd.read_csv(TABLES_DIR / 'regression_results.csv')
sensitivity_summary = pd.read_csv(TABLES_DIR / 'sensitivity_summary.csv')
display(subgroup_results.head(15))
display(kap_correlations)
display(regression_results.head(15))
display(sensitivity_summary.head(15))

## Figures

In [ ]:
figure_names = [
    'demographic_profile_panel',
    'trimmed_duration_distribution',
    'knowledge_item_lollipop',
    'attitude_diverging_likert',
    'practice_response_heatmap',
    'composite_score_distributions',
    'kap_correlation_matrix',
    'observed_vs_generated_comparison',
    'regression_effect_forest',
]
for name in figure_names:
    display(Markdown(f'### {name.replace("_", " ").title()}'))
    display(SVG(filename=str(FIGURES_DIR / f'{name}.svg')))

## Interpretation and caveats

The key takeaways below are generated after the analysis pipeline writes the final summary tables. Keep the exploratory label attached to any finding that depends on the synthetic augmentation.

In [ ]:
display(Markdown((TABLES_DIR / 'insights.md').read_text()))